# 04 — Adaptive-gap reaction bands

Use the same saved 2,000 AAPL trades. Preserve the six differences and nine reaction cells, estimate a causal movement scale, combine nearby exact prices into bounded-width bands, and score their summed evidence. Interactive controls let you inspect prefixes, gap settings and role thresholds.

No prediction model, candles or automatic parameter tuning. Every displayed view uses only its chosen prefix, but overlays final-prefix bands retrospectively; band starts are first member-price appearances, not discovery timestamps.


In [ ]:
# Basic setup. Generated data and charts must stay outside the repository.
import os
import sys
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import subprocess
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path("D:/TradingCodes/quant-research-workbench")
RUNTIME_ROOT = Path("D:/TradingML/runtimes")
SOURCE_RUN = RUNTIME_ROOT / "level-book-v8/reaction-demo/20261020T204419563339Z"
EXPECTED_SOURCE_HASH = "b40270ba8fc4055d515fa9061ce5333cce152890de23e249dbbc827e7e0ec70d"
N_EVENTS = 2000
# Explicit illustrative settings, fixed before inspecting the result.
TAU_SECONDS = 1.0          # Same 1-second reference used in our discussion; not a candle timeframe.
SCORE_THRESHOLD = 0.60     # A display/selection rule, not a calibrated confidence threshold.
WEIGHT_COUNT = 1.0 / 3     # Equal weights reproduce (normalized count + volume + time) / 3.
WEIGHT_VOLUME = 1.0 / 3
WEIGHT_TIME = 1.0 / 3
assert TAU_SECONDS > 0
assert 0 < SCORE_THRESHOLD <= 1
assert min(WEIGHT_COUNT, WEIGHT_VOLUME, WEIGHT_TIME) >= 0
assert np.isclose(WEIGHT_COUNT + WEIGHT_VOLUME + WEIGHT_TIME, 1.0)
assert REPO.is_dir() and RUNTIME_ROOT.is_dir(), "Required root unavailable"
RUN = RUNTIME_ROOT / "level-book-v8/adaptive-gap-bands" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN.mkdir(parents=True, exist_ok=False)
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(16)
print("Output directory:", RUN)
HALF_LIFE = 1000  # Nonzero price changes, not seconds or candles.
GAP_MULTIPLIER = 2.0
INITIAL_THRESHOLDS = {"support": 0.50, "resistance": 0.50, "up_continuation": 0.60, "down_continuation": 0.60}


## 1. Load and verify the original sample
This reads the saved canonical-derived snapshot, not raw SIP flatfiles. We validate its original content hash before taking the requested prefix. If it is missing or changed, stop rather than silently fetching a different sample.

The original extraction excluded delayed/unknown-clock and condition-ineligible trades. Its audit and provenance are retained. Price is stored in exact units of 1/10,000 dollar. Ordinal is metadata, not a predictor: order remains `(SIP timestamp, ordinal)`, exactly as in notebook 01. Time differences can be zero for equal timestamps.

We slice **before** calculating shifts. The first event outside the prefix must not reveal the final event's departure.

In [ ]:
# Recreate the original loader's hashed records, in their original order.
source = pl.read_parquet(SOURCE_RUN / "source_trades.parquet")
provenance = json.loads((SOURCE_RUN / "source_provenance.json").read_text(encoding="utf-8"))
source_records = source.select("ordinal", "time_us", "price_units", "volume").to_dicts()
# ClickHouse JSON encoded integral sizes as integers; Parquet promoted the
# mixed size column to Float64. Restore only that JSON representation for hashing.
for record in source_records:
    if float(record["volume"]).is_integer():
        record["volume"] = int(record["volume"])
source_hash = hashlib.sha256(json.dumps(source_records, sort_keys=True).encode()).hexdigest()
assert source_hash == provenance["sample_sha256"] == EXPECTED_SOURCE_HASH
assert source.height == 2000 and provenance["ticker"] == "AAPL"
assert provenance["day"] == "2026-08-20"
assert 3 <= N_EVENTS <= source.height
assert source["ordinal"].n_unique() == source.height
assert source.select("time_us", "ordinal").equals(source.sort("time_us", "ordinal").select("time_us", "ordinal"))
raw = source.head(N_EVENTS)
# These are the three original quantities; event and ordinal identify rows.
display(raw.select("event", "ordinal", "time", "price", "volume").head(15))
print("Selected events:", raw.height, "unique prices:", raw["price_units"].n_unique())
print("Time span:", raw["time"][0], "to", raw["time"][-1])
display(pl.DataFrame(provenance["audit"]))

## 2. Our agreed step one: six vectorized differences
For any column x:

- backward = current minus previous;
- forward = next minus current.

Time differences are seconds, price differences dollars, and volume differences changes in individual trade size. Arrival and departure are the signs of the respective price differences. A zero price difference means unchanged, not missing.

The first backward values and final forward values remain null. Forward time is the current trade's completed hold contribution. We do not add elapsed time for the pending final row or pretend its reaction is unchanged.

In [ ]:
def step_one(frame):
    """Derive adjacent-row differences on precisely the supplied prefix."""
    return frame.with_columns(
        ((pl.col("time_us") - pl.col("time_us").shift(1)) / 1e6).alias("time_backward"),
        ((pl.col("time_us").shift(-1) - pl.col("time_us")) / 1e6).alias("time_forward"),
        ((pl.col("price_units") - pl.col("price_units").shift(1)) / 10000).alias("price_backward"),
        ((pl.col("price_units").shift(-1) - pl.col("price_units")) / 10000).alias("price_forward"),
        (pl.col("volume") - pl.col("volume").shift(1)).alias("volume_backward"),
        (pl.col("volume").shift(-1) - pl.col("volume")).alias("volume_forward"),
    ).with_columns(
        pl.col("price_backward").sign().cast(pl.Int8).alias("arrival"),
        pl.col("price_forward").sign().cast(pl.Int8).alias("departure"),
    )

step1 = step_one(raw)
step_columns = ["event", "time", "price", "volume", "time_backward", "time_forward",
                "price_backward", "price_forward", "volume_backward", "volume_forward"]
display(step1.select(step_columns).head(15))
display(step1.select(step_columns).tail(3))

## 3. Group by exact price and the nine direction pairs

| Arrival / departure | Down (−1) | Unchanged (0) | Up (+1) |
|---|---|---|---|
| Down (−1) | Down continuation | Pause after decline | Support-like bounce |
| Unchanged (0) | Down after unchanged | Unchanged / unchanged | Up after unchanged |
| Up (+1) | Resistance-like rejection | Pause after rise | Up continuation |

Each trade belongs to one cell when both directions are known. There are no visit IDs. For example, down→unchanged followed by unchanged→up is **not** merged into one bounce.

Each cell stores count, current traded volume, and sums of all six differences. Dividing a difference sum by count gives its cell mean. Signed volume changes can cancel, so they remain explanatory measurements, not an automatic strength score.

Boundary rows are displayed separately and still contribute to the total activity distribution.

In [ ]:
# These are the six differences already calculated in step one.
# Listing their names lets us apply the same sum operation to each column.
difference_columns = ["time_backward", "time_forward", "price_backward", "price_forward", "volume_backward", "volume_forward"]
# Keep rows with BOTH directions known so each can be assigned to one of nine groups.
# pl.col selects a column, is_not_null checks availability, and & requires both checks.
# A direction of zero (unchanged) is valid and is retained.
complete = step1.filter(pl.col("arrival").is_not_null() & pl.col("departure").is_not_null())
# Retain rows missing either direction in a separate audit table; | means OR.
# The first arrival and final departure are unknown, not unchanged.
# Keeping them separately prevents silent loss of their count and volume.
boundary_rows = step1.filter(pl.col("arrival").is_null() | pl.col("departure").is_null())
# Group by exact integer price AND arrival AND departure.
# For example, all rows at one price with arrival=-1 and departure=+1 form one group.
# Integer price units avoid floating-point equality problems. agg calculates one row per group.
profile = complete.group_by("price_units", "arrival", "departure").agg(
    # Count the trades in this group. alias gives the result its readable column name.
    # This is a trade count, not a count of independent visits.
    pl.len().alias("trade_count"),
    # Sum CURRENT trade sizes in this group, not next-trade size or volume change.
    # Thus bounce-group volume means volume on trades immediately followed by an upward move.
    pl.col("volume").sum().alias("volume_sum"),
    # Build one sum expression per difference column; * expands the list into agg arguments.
    # Forward-time sum is the group's completed hold contribution, measured in seconds.
    # Backward time is kept separately: adding the two time sums would double-count intervals.
    # Price and volume differences are signed; positive and negative values can cancel.
    # Dividing each sum by trade_count yields that group's mean difference.
    *[pl.col(name).sum().alias(name + "_sum") for name in difference_columns],
# Close the aggregation, add a dollar-price column for display, then sort the rows.
# Grouping remains on price_units; dividing by 10,000 is presentation only.
# Sorting by price and both directions makes the table easier to inspect.
).with_columns((pl.col("price_units") / 10000).alias("price")).sort("price_units", "arrival", "departure")
# Display the first 18 aggregated rows only; the full profile remains in memory.
# This limits screen output, not the data used by subsequent calculations.
display(profile.head(18))
# Report how many original trade rows were classified and how many have unknown boundaries.
# DataFrame.height is its number of rows; these two counts should total N_EVENTS.
print("Classified rows:", complete.height, "boundary rows:", boundary_rows.height)
# Show the original event identities and known directions of the boundary rows.
# select chooses display columns; it does not modify or discard the saved boundary table.
display(boundary_rows.select("event", "price", "volume", "arrival", "departure"))

# Separately aggregate ALL selected trades by price, including the boundary rows.
# Activity totals need no next-trade outcome, so their count and volume are already known.
# This table describes overall trading concentration, not a particular reaction.
activity = raw.group_by("price_units").agg(pl.len().alias("total_trades"), pl.col("volume").sum().alias("total_volume")).sort("price_units")
# Rank prices by total trade count descending; break ties by lower price first.
# Select the first price_units value to choose one reproducible example for the matrix.
# "Busiest" here means most trades, not greatest volume or strongest support.
busiest = activity.sort(["total_trades", "price_units"], descending=[True, False])["price_units"][0]
# Explicitly create all nine (arrival, departure) pairs: each direction is -1, 0 or +1.
# The departure list repeats three times, once for each arrival direction.
# Int8 matches the direction-column types in profile so the join uses compatible keys.
grid = pl.DataFrame({"arrival": [-1,-1,-1,0,0,0,1,1,1], "departure": [-1,0,1]*3}).cast(pl.Int8)
# Filter the profile to the chosen price, then match rows by both directions.
# A LEFT join preserves all nine grid rows, even if a combination never occurred.
# An unobserved combination has null aggregate values after this join.
matrix = grid.join(profile.filter(pl.col("price_units") == busiest), on=["arrival", "departure"], how="left")
# Print the chosen price in dollars so the displayed matrix is clearly identified.
print("Nine cells at price:", busiest / 10000)
# Display directions plus count, volume and completed hold time for each of nine cells.
# Here null means NO MATCHING GROUP, so those three totals are correctly displayed as zero.
# This does not replace unknown directions or pending forward differences in the source.
# fill_null affects this display expression; the original matrix/profile remain unchanged.
display(matrix.select("arrival", "departure", pl.col("trade_count", "volume_sum", "time_forward_sum").fill_null(0)))


## 4. Estimate a movement scale that can update trade by trade
For a price change, r = abs(log(current / previous)). Update s = (1-alpha)*s + alpha*r, with alpha = 1 - 2**(-1/H). H is the half-life in **nonzero price changes**. Unchanged prices do not update s. Initialize s from the first nonzero movement.

Only previous price, s, change count and H are required to resume. The small chronological loop is the same in historical and streaming use; it does constant work per event. Vectorized differences and aggregation remain unchanged. The notebook retains a trace for teaching; a production updater needs only the final state.

The initial estimate retains weight 2**(-(changes-1)/H). We display it: a short prefix with few changes is immature. This is an explanatory status, not a hidden selection gate. An average is sensitive to outliers; no clipping is silently introduced. Corporate actions/data-policy changes need their own explicit reset/version policy outside this single-session demonstration.

In [ ]:
def movement_scale(prices, half_life, state=None):
    """Exact resumable event-time recurrence; state is small and JSON-serializable."""
    assert half_life > 0 and np.isfinite(half_life)
    state = dict(previous=None, scale=None, changes=0, half_life=float(half_life)) if state is None else dict(state)
    assert state["half_life"] == half_life, "Changing H requires recomputing the selected prefix"
    alpha = 1.0 - 2.0 ** (-1.0 / half_life)
    trace = []
    for price in prices:
        price = int(price)
        assert price > 0
        if state["previous"] is not None and price != state["previous"]:
            # log1p is numerically stable for small relative changes.
            movement = abs(np.log1p((price - state["previous"]) / state["previous"]))
            state["scale"] = float(movement if state["scale"] is None else
                                   (1-alpha)*state["scale"] + alpha*movement)
            state["changes"] += 1
        state["previous"] = price
        trace.append(state["scale"])
    return state, trace

scale_state, scale_trace = movement_scale(raw["price_units"], HALF_LIFE)
initial_weight = None if not scale_state["changes"] else 2 ** (-(scale_state["changes"]-1)/HALF_LIFE)
print("Final state:", scale_state)
print("Weight still assigned to the first movement:", initial_weight)
display(raw.select("event", "price").with_columns(pl.Series("movement_scale", scale_trace)).tail(10))

## 5. Form adaptive bands without unlimited chaining
For a band's lowest member price p, calculate an allowed width g = p * expm1(k*s). Round UP to the source price grid of 1/10,000 dollar; that precision is not a claim about an exchange tick size.

Sort unique observed prices. The lowest unassigned price starts a band. Add following prices while their distance from this **fixed band start** is <= g. Otherwise start another band. This explicit maximum-span rule prevents 100.00→100.02→100.04→... from chaining into one unlimited band. The gap is frozen for each construction pass; it is not updated while traversing prices.

This is a greedy left-to-right partition, not an optimized density model. Its result depends on the start anchor. It uses one scale per selected prefix, and O(K) work after sorting K unique prices. Streaming scale updates remain O(1); membership is rebuilt only when requesting a band-building pass, not after every raw trade. IDs are local to that pass and may change when the prefix or parameters change.

All exact prices are assigned once. Original arrivals/departures are never recalculated from banded prices. Observed min/max define the drawn band; the allowed width is a limit, not artificially added padding. Singleton bands draw as lines.

In [ ]:
def band_membership(prices, scale, multiplier):
    """Assign sorted unique integer prices, with width capped from each band's anchor."""
    assert multiplier > 0 and np.isfinite(multiplier)
    ordered = np.unique(np.asarray(prices, dtype=np.int64))
    rows = []
    anchor, allowed, band_id = None, 0, -1
    for price in ordered:
        if anchor is None or int(price) - anchor > allowed:
            band_id += 1
            anchor = int(price)
            # No observed movement means no estimated merging distance.
            allowed = 0 if scale is None else max(1, int(np.ceil(anchor * np.expm1(multiplier * scale))))
        rows.append(dict(price_units=int(price), band_id=band_id, anchor_units=anchor, allowed_width_units=allowed))
    return pl.DataFrame(rows)

membership = band_membership(raw["price_units"], scale_state["scale"], GAP_MULTIPLIER)
display(membership.with_columns((pl.col("allowed_width_units")/10000).alias("allowed_width_dollars")).head(20))
print("Exact prices:", membership.height, "bands:", membership["band_id"].n_unique())

## 6. Aggregate original evidence first, then score bands
Join each exact price to its band. Sum each of the nine cells' trade count, current volume and all six difference sums. Do not average normalized price scores.

Band center is the volume-weighted mean of **all current trade prices** in that band. It is independent of role. Band first_event is the first occurrence of any member price. Count/volume/time reaction scores use the same four cells and formula as notebook 03, but maxima are now computed across bands separately for each role.

Overall occupancy sums all known forward intervals, including unchanged cells and any first boundary row whose next trade is known. It remains separate from directional scores. Missing directional cells score zero; their mean delay remains unknown.

In [ ]:
# Define the four roles and their time interpretation explicitly.
roles = pl.DataFrame({
    "role": ["support", "resistance", "up_continuation", "down_continuation"],
    "arrival": [-1, 1, 1, -1],
    "departure": [1, -1, 1, -1],
    "time_preference": ["quick", "quick", "long", "long"],
}).with_columns(pl.col("arrival", "departure").cast(pl.Int8))

def score_bands(price_activity, reaction_profile, tau, weights):
    """Score every band/role, including empty cells. No neighbor or peak tests."""
    assert np.isfinite(tau) and tau > 0
    assert len(weights) == 3 and min(weights) >= 0 and np.isclose(sum(weights), 1)
    # Cross join creates all four roles at every band, even when a cell is absent.
    table = price_activity.join(roles, how="cross").join(
        reaction_profile.select("band_id", "arrival", "departure", "trade_count", "volume_sum", "time_forward_sum"),
        on=["band_id", "arrival", "departure"], how="left",
    ).with_columns(
        # Missing group means zero observations/totals, not an unknown source direction.
        pl.col("trade_count", "volume_sum", "time_forward_sum").fill_null(0),
    ).with_columns(
        # over('role') computes each role's maximum across bands, without merging roles.
        pl.col("trade_count").max().over("role").alias("max_count"),
        pl.col("volume_sum").max().over("role").alias("max_volume"),
        # No observations means no known mean delay; leave it null rather than set it to zero.
        pl.when(pl.col("trade_count") > 0).then(pl.col("time_forward_sum") / pl.col("trade_count"))
          .otherwise(None).alias("mean_delay_seconds"),
    ).with_columns(
        # If a role has no evidence anywhere, its count/volume components are zero.
        pl.when(pl.col("max_count") > 0).then(pl.col("trade_count") / pl.col("max_count"))
          .otherwise(0.0).alias("count_component"),
        pl.when(pl.col("max_volume") > 0).then(pl.col("volume_sum") / pl.col("max_volume"))
          .otherwise(0.0).alias("volume_component"),
        # Select QUICK for reversal roles and LONG for continuation roles.
        pl.when(pl.col("trade_count") == 0).then(0.0)
          .when(pl.col("time_preference") == "quick").then(tau / (tau + pl.col("mean_delay_seconds")))
          .otherwise(pl.col("mean_delay_seconds") / (tau + pl.col("mean_delay_seconds"))).alias("time_component"),
    ).with_columns(
        # Expose the contribution of each term so the final score is easy to audit.
        (weights[0] * pl.col("count_component")).alias("count_contribution"),
        (weights[1] * pl.col("volume_component")).alias("volume_contribution"),
        (weights[2] * pl.col("time_component")).alias("time_contribution"),
    ).with_columns(
        # An unobserved cell cannot earn a score from a time bonus.
        pl.when(pl.col("trade_count") > 0)
          .then(pl.col("count_contribution") + pl.col("volume_contribution") + pl.col("time_contribution"))
          .otherwise(0.0).alias("score"),
    )
    return table.sort(["role", "score", "band_id"], descending=[False, True, False])


In [ ]:
def analyze_prefix(frame, half_life, multiplier, tau):
    """One bounded as-of analysis. No data outside frame participates."""
    derived = step_one(frame)
    state, trace = movement_scale(frame["price_units"], half_life)
    mapping = band_membership(frame["price_units"], state["scale"], multiplier)
    assigned = derived.join(mapping, on="price_units", how="left")
    # Geometry and overall activity use all trades, including boundary rows.
    geometry = assigned.group_by("band_id").agg(
        pl.col("price_units").min().alias("lower_units"),
        pl.col("price_units").max().alias("upper_units"),
        pl.col("price_units").n_unique().alias("member_prices"),
        pl.col("event").min().alias("first_event"),
        pl.col("anchor_units").first(), pl.col("allowed_width_units").first(),
        pl.len().alias("total_trades"), pl.col("volume").sum().alias("total_volume"),
        ((pl.col("price_units") * pl.col("volume")).sum() / pl.col("volume").sum() / 10000).alias("center"),
        pl.col("time_forward").sum().alias("occupancy_seconds"),
    ).with_columns((pl.col("lower_units")/10000).alias("lower"), (pl.col("upper_units")/10000).alias("upper")).sort("band_id")
    known = assigned.filter(pl.col("arrival").is_not_null() & pl.col("departure").is_not_null())
    grouped = known.group_by("band_id", "arrival", "departure").agg(
        pl.len().alias("trade_count"), pl.col("volume").sum().alias("volume_sum"),
        *[pl.col(name).sum().alias(name + "_sum") for name in difference_columns],
    )
    scored = score_bands(geometry.select("band_id", "total_trades", "total_volume"), grouped,
                        tau, (WEIGHT_COUNT, WEIGHT_VOLUME, WEIGHT_TIME))
    scored = scored.join(geometry.drop("total_trades", "total_volume"), on="band_id", how="left")
    return dict(frame=frame, derived=derived, state=state, trace=trace, mapping=mapping,
                assigned=assigned, geometry=geometry, profile=grouped, scores=scored,
                boundary=assigned.filter(pl.col("arrival").is_null() | pl.col("departure").is_null()))

analysis = analyze_prefix(raw, HALF_LIFE, GAP_MULTIPLIER, TAU_SECONDS)
display(analysis["geometry"])
display(analysis["profile"].sort("band_id", "arrival", "departure").head(18))
display(analysis["scores"].select("band_id", "lower", "upper", "center", "role", "trade_count", "volume_sum",
                                 "mean_delay_seconds", "count_component", "volume_component", "time_component", "score").head(20))

## 7. Interactive chart: inspect the same sample at different cutoffs
Controls: event count, nonzero-change half-life H, gap multiplier k, time reference tau, and four independent role thresholds. Support/resistance default to 0.50 as requested. Release a slider to redraw. Threshold-only changes reuse the current band analysis; changing the prefix/H/k/tau recomputes from that prefix.

Each role has its own price panel so overlapping support/resistance/continuation scores remain readable. Shading spans observed member prices, begins at the first member-price appearance and ends at the selected cutoff. The center line is volume-weighted. Labels show band bounds and score.

All these are final-prefix overlays, not historical signals. A small prefix with H=1000 will usually have immature movement-scale evidence. The status reports that explicitly. The latest view, settings and underlying band tables are saved outside the repository. A live kernel and ipywidgets are required.

In [ ]:
import ipywidgets as widgets
role_colors = {"support": "#16803c", "resistance": "#c53030", "up_continuation": "#2563eb", "down_continuation": "#9333ea"}
role_labels = {"support": "Support", "resistance": "Resistance", "up_continuation": "Up continuation", "down_continuation": "Down continuation"}
style = {"description_width": "145px"}
layout = widgets.Layout(width="450px")
controls = {
    "events": widgets.IntSlider(value=N_EVENTS, min=3, max=source.height, step=1, description="Events", continuous_update=False, style=style, layout=layout),
    "half_life": widgets.IntSlider(value=HALF_LIFE, min=10, max=5000, step=10, description="Change half-life", continuous_update=False, style=style, layout=layout),
    "multiplier": widgets.FloatSlider(value=GAP_MULTIPLIER, min=0.25, max=10, step=0.25, description="Gap multiplier", continuous_update=False, style=style, layout=layout),
    "tau": widgets.FloatLogSlider(value=TAU_SECONDS, base=10, min=-4, max=1, step=0.25, description="Time reference (s)", readout_format=".4f", continuous_update=False, style=style, layout=layout),
}
for role in role_colors:
    controls[role] = widgets.FloatSlider(value=INITIAL_THRESHOLDS[role], min=0, max=1, step=0.01,
        description=role_labels[role], readout_format=".2f", continuous_update=False, style=style, layout=layout)
# Keep one cached analysis only. Threshold changes do not rerun the grouping.
view_cache = {}; band_chart_state = {}

def draw_bands(events, half_life, multiplier, tau, support, resistance, up_continuation, down_continuation):
    key = (events, half_life, multiplier, tau)
    if view_cache.get("key") != key:
        view_cache.clear()
        view_cache.update(key=key, analysis=analyze_prefix(source.head(events), half_life, multiplier, tau))
    result = view_cache["analysis"]
    thresholds = dict(support=support, resistance=resistance, up_continuation=up_continuation, down_continuation=down_continuation)
    cutoff = int(result["frame"]["event"].max())
    state = result["state"]
    remaining = None if state["changes"] == 0 else 2**(-(state["changes"]-1)/half_life)
    print(f"{events} trades | {result['mapping'].height} exact prices -> {result['geometry'].height} bands | {state['changes']} nonzero changes")
    print("Initial-movement weight:", remaining, "|", "immature (< H changes)" if state["changes"] < half_life else "at least H changes observed")
    selected = result["scores"].join(pl.DataFrame({"role": list(thresholds), "threshold": list(thresholds.values())}), on="role").filter(
        (pl.col("trade_count") > 0) & (pl.col("score") >= pl.col("threshold")))
    maximum = max([selected.filter(pl.col("role") == r).height for r in role_colors] + [1])
    fig, axes = plt.subplots(4, 1, figsize=(14, 4*max(3.2, 0.30*maximum)), sharex=True)
    fig.subplots_adjust(right=0.76, left=0.08, bottom=0.05, top=0.95, hspace=0.28)
    for ax, role in zip(axes, role_colors):
        ax.plot(result["frame"]["event"], result["frame"]["price"], color="black", linewidth=0.8)
        bands = selected.filter(pl.col("role") == role).sort("lower_units")
        lo, hi = ax.get_ylim()
        labels = np.linspace(lo + .04*(hi-lo), hi - .04*(hi-lo), bands.height)
        for row, label_y in zip(bands.iter_rows(named=True), labels):
            color = role_colors[role]
            ax.fill_between([row["first_event"], cutoff], row["lower"], row["upper"], color=color, alpha=0.16)
            ax.hlines([row["lower"], row["upper"]], row["first_event"], cutoff, color=color, alpha=0.5, linewidth=0.6)
            ax.hlines(row["center"], row["first_event"], cutoff, color=color, linewidth=0.9)
            ax.scatter([row["first_event"]], [row["center"]], color=color, s=8)
            ax.annotate(f"{row['lower']:.4f}-{row['upper']:.4f} | {row['score']:.3f}",
                xy=(cutoff,row["center"]), xycoords="data", xytext=(1.02,label_y), textcoords=ax.get_yaxis_transform(),
                fontsize=8,color=color,va="center",annotation_clip=False,
                arrowprops=dict(arrowstyle="-",color=color,linewidth=.5,relpos=(0,.5)))
        ax.set(title=f"{role_labels[role]} >= {thresholds[role]:.2f}: {bands.height} bands", ylabel="Trade price ($)")
        ax.grid(alpha=.12)
    axes[-1].set_xlabel("Event number; bands start at first member-price appearance")
    fig.suptitle("Adaptive-gap bands — final-prefix evidence, not historical signals")
    settings = dict(events=events, half_life=half_life, multiplier=multiplier, tau=tau, thresholds=thresholds,
                    source_sha256=source_hash, movement_state=state)
    fig.savefig(RUN / "adaptive_bands.png", dpi=140, bbox_inches="tight")
    plt.show(); plt.close(fig)
    # Save exactly the current view, not only the initial default settings.
    result["mapping"].write_parquet(RUN / "current_membership.parquet")
    result["geometry"].write_parquet(RUN / "current_bands.parquet")
    result["profile"].write_parquet(RUN / "current_nine_cell_profile.parquet")
    result["scores"].write_parquet(RUN / "current_scores.parquet")
    selected.write_parquet(RUN / "current_selected_bands.parquet")
    (RUN / "view_settings.json").write_text(json.dumps(settings,indent=2),encoding="utf-8")
    band_chart_state.update(result=result, selected=selected, settings=settings, renders=band_chart_state.get("renders",0)+1)
    display(selected.select("band_id","role","lower","upper","center","first_event","trade_count","volume_sum","score").sort("role","lower"))

interactive_bands = widgets.interactive_output(draw_bands, controls)
display(widgets.VBox([widgets.HBox([controls["events"], controls["half_life"]]),
                     widgets.HBox([controls["multiplier"], controls["tau"]]),
                     widgets.HBox([controls["support"],controls["resistance"]]),
                     widgets.HBox([controls["up_continuation"],controls["down_continuation"]]), interactive_bands]))

## 8. Validate the recurrence, grouping and conservation
Tests below check exact state restoration, unchanged trades, bounded width, no chaining, every price assigned once, total evidence conservation, mean-delay recomputation, score bounds and prefix isolation. These establish calculation behavior, not useful future support/resistance. No campaign or production database is changed.

In [ ]:
# A resumed updater must reach exactly the uninterrupted state.
prices = raw["price_units"].to_numpy()
full, _ = movement_scale(prices, HALF_LIFE)
part, _ = movement_scale(prices[:1000], HALF_LIFE)
resumed, _ = movement_scale(prices[1000:], HALF_LIFE, part)
assert full == resumed
flat, _ = movement_scale([100,100,100], HALF_LIFE)
assert flat["scale"] is None and flat["changes"] == 0
single, _ = movement_scale([100,101,101], HALF_LIFE)
assert np.isclose(single["scale"], abs(np.log(101/100))) and single["changes"] == 1
# This scale permits two integer units from each anchor: the chain must split.
chain = band_membership([100,101,102,103,104,105], np.log1p(.019), 1.)
assert chain["band_id"].to_list() == [0,0,0,1,1,1]
assert band_membership([100,100], None, 2.)["band_id"].to_list() == [0]
result = analysis
assert result["mapping"]["price_units"].n_unique() == raw["price_units"].n_unique()
assert result["assigned"].height == raw.height
geometry = result["geometry"]
assert ((geometry["upper_units"]-geometry["lower_units"]) <= geometry["allowed_width_units"]).all()
assert geometry["total_trades"].sum() == raw.height
assert np.isclose(geometry["total_volume"].sum(),raw["volume"].sum())
assert result["profile"]["trade_count"].sum() + result["boundary"].height == raw.height
assert np.isclose(result["profile"]["volume_sum"].sum()+result["boundary"]["volume"].sum(),raw["volume"].sum())
assert np.isclose(geometry["occupancy_seconds"].sum(),(raw["time_us"][-1]-raw["time_us"][0])/1e6)
for col in difference_columns:
    assert np.isclose(result["profile"][col+"_sum"].sum(),complete[col].sum())
observed = result["scores"].filter(pl.col("trade_count")>0)
assert np.allclose(observed["mean_delay_seconds"],observed["time_forward_sum"]/observed["trade_count"])
assert result["scores"]["score"].min() >= 0 and result["scores"]["score"].max() <= 1+1e-12
# Altering only the suffix cannot affect a 200-event analysis.
changed = source.with_columns(pl.when(pl.col("event")>200).then(pl.col("price_units")+100000).otherwise(pl.col("price_units")).alias("price_units"))
a = analyze_prefix(source.head(200), HALF_LIFE, GAP_MULTIPLIER, TAU_SECONDS)
b = analyze_prefix(changed.head(200), HALF_LIFE, GAP_MULTIPLIER, TAU_SECONDS)
assert a["state"] == b["state"] and a["mapping"].equals(b["mapping"])
assert a["scores"].sort("band_id","role").equals(b["scores"].sort("band_id","role"))
print("PASS: recurrence/resume, flat prices, no chaining, bounded span, count/volume/time conservation, delay, scores, prefix isolation")

NOTEBOOK = REPO / "research/level_book/v8/04_adaptive_gap_reaction_bands.ipynb"
manifest = dict(source_sha256=source_hash, default_events=N_EVENTS, default_half_life=HALF_LIFE,
    default_multiplier=GAP_MULTIPLIER, default_tau=TAU_SECONDS, thresholds=INITIAL_THRESHOLDS,
    exact_prices=result["mapping"].height, bands=geometry.height, movement_state=full, checks="passed",
    notebook_sha256=hashlib.sha256(NOTEBOOK.read_bytes()).hexdigest(),
    git_commit=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip(),
    python=sys.version,numpy=np.__version__,polars=pl.__version__)
(RUN / "manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
(RUN / "source_provenance.json").write_text(json.dumps(provenance,indent=2),encoding="utf-8")
print(json.dumps(manifest,indent=2))